# 03 — Features de risque et scoring

Objectif : construire un score de risque reproductible pour chaque subscriber à partir de la base nettoyée.

Le score repose uniquement sur des signaux observables dans les données nettoyées, puis il est exporté en CSV.

## Périmètre retenu

On utilise surtout les paiements, les réclamations, les memberships et la récence d’activité.

On ne s’appuie pas sur le pays ni sur les statuts numériques non documentés comme signal métier principal.

In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd
import numpy as np

# Base nettoyée issue de l'étape 2
db_path = Path("data/processed/risk_monitor_clean.sqlite")
assert db_path.exists(), f"Fichier introuvable : {db_path}"

conn = sqlite3.connect(db_path)

users = pd.read_sql_query("SELECT * FROM users;", conn)
subscriptions = pd.read_sql_query("SELECT * FROM subscriptions;", conn)
memberships = pd.read_sql_query("SELECT * FROM memberships;", conn)
payments = pd.read_sql_query("SELECT * FROM payments;", conn)
complaints = pd.read_sql_query("SELECT * FROM complaints;", conn)

conn.close()

## Fonctions utilitaires

On prépare quelques fonctions simples pour gérer les dates, les divisions sûres et les normalisations robustes.

Le but est de garder un calcul déterministe et lisible.

In [2]:
def to_utc(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series, utc=True, errors="coerce")

def robust_minmax(series: pd.Series, lower_q: float = 0.05, upper_q: float = 0.95) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce").fillna(0.0)
    lo = s.quantile(lower_q)
    hi = s.quantile(upper_q)
    if pd.isna(lo) or pd.isna(hi) or hi <= lo:
        return pd.Series(0.0, index=s.index)
    return ((s - lo) / (hi - lo)).clip(0, 1)

def safe_div(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    num = pd.to_numeric(numerator, errors="coerce").fillna(0.0)
    den = pd.to_numeric(denominator, errors="coerce").replace(0, np.nan)
    return (num / den).fillna(0.0)

def days_between(later: pd.Series, earlier: pd.Series) -> pd.Series:
    delta = later - earlier
    return delta.dt.total_seconds().div(86400).fillna(0.0)

def clip_days(series: pd.Series, max_days: float = 365.0) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce").fillna(0.0)
    return s.clip(lower=0.0, upper=max_days)

def canonical_complaint_type(value: str) -> str:
    if pd.isna(value):
        return "other"
    s = str(value).strip().lower()
    mapping = {
        "acces refuse": "access_denied",
        "access denied": "access_denied",
        "billing issue": "billing_issue",
        "billing_issue": "billing_issue",
        "fraud suspicion": "fraud_suspicion",
        "fraud_suspicion": "fraud_suspicion",
        "owner unresponsive": "owner_unresponsive",
        "owner_unresponsive": "owner_unresponsive",
        "subscription inactive": "subscription_inactive",
        "subscription_inactive": "subscription_inactive",
        "wrong credentials": "wrong_credentials",
        "wrong_credentials": "wrong_credentials",
        "other": "other",
    }
    return mapping.get(s, s)

## Date de référence

La date de référence est dérivée des données elles-mêmes pour garantir un scoring reproductible.

On prend la date la plus récente observée dans les tables, puis on ajoute un jour.

In [3]:
# Conversion des colonnes temporelles utiles
for df, cols in [
    (users, ["last_seen_at_utc"]),
    (subscriptions, ["created_at_utc"]),
    (memberships, ["joined_at_utc", "left_at_utc"]),
    (payments, ["created_at_utc", "captured_at_utc"]),
    (complaints, ["created_at_utc", "resolved_at_utc"]),
]:
    for col in cols:
        df[col] = to_utc(df[col])

# On exclut signup_at_utc du calcul de référence, car une valeur future anormale a été détectée.
date_series = pd.concat([
    users["last_seen_at_utc"],
    subscriptions["created_at_utc"],
    memberships["joined_at_utc"],
    memberships["left_at_utc"],
    payments["created_at_utc"],
    payments["captured_at_utc"],
    complaints["created_at_utc"],
    complaints["resolved_at_utc"],
], ignore_index=True).dropna()

reference_date = date_series.max().normalize() + pd.Timedelta(days=1)
reference_date

Timestamp('2026-04-11 00:00:00+0000', tz='UTC')

## Features paiements

Les paiements sont le signal le plus direct de risque opérationnel.

On compte les échecs, litiges, remboursements, attentes et annulations, puis on mesure la récence du dernier paiement avec une fenêtre plus large pour éviter une saturation trop rapide.

In [4]:
PAYMENT_STATUS_WEIGHTS = {
    "succeeded": 0.0,
    "failed": 1.0,
    "disputed": 1.5,
    "refunded": 0.8,
    "pending": 0.3,
    "canceled": 0.2,
}

p = payments.copy()
p["status_key"] = p["status"].astype(str).str.lower()

for st in PAYMENT_STATUS_WEIGHTS.keys():
    p[f"is_{st}"] = (p["status_key"] == st).astype(int)

payment_last = p.groupby("user_id", as_index=False).agg(
    payment_last_at=("created_at_utc", "max")
)

payment_agg = p.groupby("user_id", as_index=False).agg(
    payment_total_count=("id", "count"),
    payment_success_count=("is_succeeded", "sum"),
    payment_failed_count=("is_failed", "sum"),
    payment_disputed_count=("is_disputed", "sum"),
    payment_refunded_count=("is_refunded", "sum"),
    payment_pending_count=("is_pending", "sum"),
    payment_canceled_count=("is_canceled", "sum"),
    payment_total_amount_cents=("amount_cents", "sum"),
    payment_avg_amount_cents=("amount_cents", "mean"),
    payment_avg_fee_cents=("fee_cents", "mean"),
    payment_distinct_subscription_count=("subscription_id", "nunique"),
)

payments_with_brand = p.merge(
    subscriptions[["id", "brand"]],
    left_on="subscription_id",
    right_on="id",
    how="left",
    suffixes=("", "_sub"),
)

brand_agg = payments_with_brand.groupby("user_id", as_index=False).agg(
    payment_distinct_brand_count=("brand", "nunique")
)

payment_agg = payment_agg.merge(brand_agg, on="user_id", how="left")
payment_agg = payment_agg.merge(payment_last, on="user_id", how="left")

valid_user_ids = set(users["id"].dropna().astype(int))
payment_agg = payment_agg[payment_agg["user_id"].isin(valid_user_ids)].copy()

payment_agg["payment_issue_weight_sum"] = (
    1.0 * payment_agg["payment_failed_count"]
    + 1.5 * payment_agg["payment_disputed_count"]
    + 0.8 * payment_agg["payment_refunded_count"]
    + 0.3 * payment_agg["payment_pending_count"]
    + 0.2 * payment_agg["payment_canceled_count"]
)

payment_agg["payment_issue_rate"] = safe_div(
    payment_agg["payment_issue_weight_sum"],
    payment_agg["payment_total_count"],
)

payment_agg["days_since_last_payment_raw"] = days_between(
    pd.Series(reference_date, index=payment_agg.index),
    payment_agg["payment_last_at"],
)

payment_agg["days_since_last_payment"] = clip_days(
    payment_agg["days_since_last_payment_raw"],
    1095,
)

payment_agg["payment_recent_event_risk_raw"] = np.where(
    payment_agg["payment_total_count"] > 0,
    1 / (1 + payment_agg["days_since_last_payment"] / 30.0),
    0.0,
)

payment_agg["payment_recent_event_risk_raw"] = payment_agg[
    "payment_recent_event_risk_raw"
].clip(0, 1)

payment_agg.head(3)

,user_id,payment_total_count,payment_success_count,payment_failed_count,payment_disputed_count,payment_refunded_count,payment_pending_count,payment_canceled_count,payment_total_amount_cents,payment_avg_amount_cents,payment_avg_fee_cents,payment_distinct_subscription_count,payment_distinct_brand_count,payment_last_at,payment_issue_weight_sum,payment_issue_rate,days_since_last_payment_raw,days_since_last_payment,payment_recent_event_risk_raw
2,1,2,0,2,0,0,0,0,1325,662.500000,67.500000,1,1,2024-09-21 07:35:43+00:00,2.0,1.000000,566.683530,566.683530,0.050278
3,6,1,1,0,0,0,0,0,0,0.000000,0.000000,1,1,2024-11-29 10:00:00+00:00,0.0,0.000000,497.583333,497.583333,0.056863
4,10,13,5,6,0,1,1,0,9148,703.692308,78.307692,1,1,2022-07-10 18:58:29+00:00,7.1,0.546154,1370.209387,1095.000000,0.026667


### Validation des features paiements

Les agrégats paiements sont cohérents et les `user_id` orphelins ont bien été exclus de la base de travail.

La récence a été rendue plus progressive pour éviter une saturation trop rapide du signal. La version brute est conservée dans `days_since_last_payment_raw`, tandis que la version bornée et la composante de risque restent exploitables pour le score final.

## Features memberships

Cette étape mesure la stabilité des parcours d’abonnement.

On calcule :
- le nombre total de memberships,
- le nombre de sorties,
- le churn,
- la durée moyenne,
- le nombre de memberships courts,
- la diversité des abonnements et des marques,
- la récence de la dernière fin de membership.

In [5]:
m = memberships.copy()
m = m.merge(
    subscriptions[["id", "brand"]],
    left_on="subscription_id",
    right_on="id",
    how="left",
    suffixes=("", "_sub"),
)

m["joined_at_utc"] = to_utc(m["joined_at_utc"])
m["left_at_utc"] = to_utc(m["left_at_utc"])

m["membership_end_at"] = m["left_at_utc"].fillna(reference_date)
m["membership_duration_days"] = days_between(m["membership_end_at"], m["joined_at_utc"])
m["membership_duration_days"] = m["membership_duration_days"].clip(lower=0)

m["is_exit"] = m["left_at_utc"].notna().astype(int)
m["is_current"] = m["left_at_utc"].isna().astype(int)
m["is_short_membership"] = (m["membership_duration_days"] < 30).astype(int)

membership_agg = m.groupby("user_id", as_index=False).agg(
    membership_total_count=("id", "count"),
    membership_exit_count=("is_exit", "sum"),
    membership_current_count=("is_current", "sum"),
    short_membership_count=("is_short_membership", "sum"),
    avg_membership_duration_days=("membership_duration_days", "mean"),
    median_membership_duration_days=("membership_duration_days", "median"),
    distinct_subscription_count=("subscription_id", "nunique"),
    distinct_brand_count=("brand", "nunique"),
    last_membership_end_at=("membership_end_at", "max"),
)

membership_agg["membership_churn_rate"] = safe_div(
    membership_agg["membership_exit_count"],
    membership_agg["membership_total_count"],
)

membership_agg["brand_switch_rate"] = safe_div(
    (membership_agg["distinct_brand_count"] - 1).clip(lower=0),
    membership_agg["membership_total_count"],
)

membership_agg["days_since_last_membership_end_raw"] = days_between(
    pd.Series(reference_date, index=membership_agg.index),
    membership_agg["last_membership_end_at"],
)

membership_agg["days_since_last_membership_end"] = clip_days(
    membership_agg["days_since_last_membership_end_raw"],
    1095,
)

membership_agg["membership_recent_event_risk_raw"] = np.where(
    membership_agg["membership_total_count"] > 0,
    1 / (1 + membership_agg["days_since_last_membership_end"] / 30.0),
    0.0,
)

membership_agg["membership_recent_event_risk_raw"] = membership_agg[
    "membership_recent_event_risk_raw"
].clip(0, 1)

membership_agg.head(3)

,user_id,membership_total_count,membership_exit_count,membership_current_count,short_membership_count,avg_membership_duration_days,median_membership_duration_days,distinct_subscription_count,distinct_brand_count,last_membership_end_at,membership_churn_rate,brand_switch_rate,days_since_last_membership_end_raw,days_since_last_membership_end,membership_recent_event_risk_raw
0,1,2,0,2,0,715.320932,715.320932,2,2,2026-04-11 00:00:00+00:00,0.0,0.5,0.0,0.0,1.0
1,10,1,0,1,0,1733.209387,1733.209387,1,1,2026-04-11 00:00:00+00:00,0.0,0.0,0.0,0.0,1.0
2,11,1,0,1,0,717.526435,717.526435,1,1,2026-04-11 00:00:00+00:00,0.0,0.0,0.0,0.0,1.0


### Validation des features memberships

Les features memberships sont cohérentes.

La date de fin de membership est remplacée par la date de référence lorsque le membership est encore actif, ce qui explique que `last_membership_end_at` et `days_since_last_membership_end` puissent prendre respectivement la valeur de référence ou zéro. Cette convention est volontaire et permet de mesurer correctement la durée et le churn.

## Features réclamations

Les réclamations portent un signal fort de risque opérationnel.

On distingue le rôle de l’utilisateur :
- comme reporter,
- comme target.

On pondère aussi le type et le statut de la réclamation pour capter la gravité, puis on ajoute une composante de récence plus progressive pour éviter une saturation trop rapide.

In [6]:
# Préparation des réclamations avec un vocabulaire canonique
c = complaints.copy()

# Sécurisation des dates
c["created_at_utc"] = pd.to_datetime(c["created_at_utc"], utc=True, errors="coerce")
c["resolved_at_utc"] = pd.to_datetime(c["resolved_at_utc"], utc=True, errors="coerce")

# Canonicalisation des types et statuts
c["type_key"] = c["type"].apply(canonical_complaint_type)
c["status_key"] = c["status"].astype(str).str.lower()

# Pondérations métier
COMPLAINT_TYPE_WEIGHTS = {
    "access_denied": 1.2,
    "fraud_suspicion": 1.5,
    "owner_unresponsive": 1.1,
    "subscription_inactive": 1.0,
    "billing_issue": 0.8,
    "wrong_credentials": 0.7,
    "other": 0.5,
}

COMPLAINT_STATUS_WEIGHTS = {
    "open": 1.0,
    "in_progress": 0.9,
    "escalated": 1.3,
    "resolved": 0.5,
    "closed": 0.4,
}

# Sévérité par réclamation
c["complaint_type_weight"] = c["type_key"].map(COMPLAINT_TYPE_WEIGHTS).fillna(0.5)
c["complaint_status_weight"] = c["status_key"].map(COMPLAINT_STATUS_WEIGHTS).fillna(0.5)
c["complaint_severity"] = c["complaint_type_weight"] * c["complaint_status_weight"]

# Flags simples
c["is_open"] = c["status_key"].isin(["open", "in_progress"]).astype(int)
c["is_escalated"] = (c["status_key"] == "escalated").astype(int)
c["is_resolved"] = (c["status_key"] == "resolved").astype(int)

# Agrégats côté reporter
reporter_agg = c.groupby("reporter_id", as_index=False).agg(
    complaint_reporter_count=("id", "count"),
    complaint_reporter_open_count=("is_open", "sum"),
    complaint_reporter_escalated_count=("is_escalated", "sum"),
    complaint_reporter_resolved_count=("is_resolved", "sum"),
    complaint_reporter_severity_sum=("complaint_severity", "sum"),
    complaint_reporter_last_at=("created_at_utc", "max"),
).rename(columns={"reporter_id": "user_id"})

# Agrégats côté target
target_agg = c.groupby("target_id", as_index=False).agg(
    complaint_target_count=("id", "count"),
    complaint_target_open_count=("is_open", "sum"),
    complaint_target_escalated_count=("is_escalated", "sum"),
    complaint_target_resolved_count=("is_resolved", "sum"),
    complaint_target_severity_sum=("complaint_severity", "sum"),
    complaint_target_last_at=("created_at_utc", "max"),
).rename(columns={"target_id": "user_id"})

# Fusion des deux rôles
complaint_agg = reporter_agg.merge(target_agg, on="user_id", how="outer")

# Sécurisation des colonnes temporelles après fusion
complaint_agg["complaint_reporter_last_at"] = pd.to_datetime(
    complaint_agg["complaint_reporter_last_at"], utc=True, errors="coerce"
)
complaint_agg["complaint_target_last_at"] = pd.to_datetime(
    complaint_agg["complaint_target_last_at"], utc=True, errors="coerce"
)

# Remplissage uniquement des colonnes numériques
numeric_cols = complaint_agg.select_dtypes(include=["number"]).columns.tolist()
for col in numeric_cols:
    complaint_agg[col] = complaint_agg[col].fillna(0)

# Totaux consolidés
complaint_agg["complaint_total_count"] = (
    complaint_agg["complaint_reporter_count"] + complaint_agg["complaint_target_count"]
)
complaint_agg["complaint_open_count"] = (
    complaint_agg["complaint_reporter_open_count"] + complaint_agg["complaint_target_open_count"]
)
complaint_agg["complaint_escalated_count"] = (
    complaint_agg["complaint_reporter_escalated_count"] + complaint_agg["complaint_target_escalated_count"]
)
complaint_agg["complaint_resolved_count"] = (
    complaint_agg["complaint_reporter_resolved_count"] + complaint_agg["complaint_target_resolved_count"]
)
complaint_agg["complaint_total_severity"] = (
    0.8 * complaint_agg["complaint_reporter_severity_sum"]
    + 1.2 * complaint_agg["complaint_target_severity_sum"]
)

# Dernière réclamation impliquant l'utilisateur
complaint_agg["last_complaint_at"] = complaint_agg[
    ["complaint_reporter_last_at", "complaint_target_last_at"]
].max(axis=1)

# Récence brute
complaint_agg["days_since_last_complaint_raw"] = days_between(
    pd.Series(reference_date, index=complaint_agg.index),
    complaint_agg["last_complaint_at"],
)

# Récence bornée sur une fenêtre plus large pour garder de la nuance
complaint_agg["days_since_last_complaint"] = clip_days(
    complaint_agg["days_since_last_complaint_raw"],
    1095,
)

# Composante de récence progressive
complaint_agg["complaint_recent_event_risk_raw"] = np.where(
    complaint_agg["complaint_total_count"] > 0,
    1 / (1 + complaint_agg["days_since_last_complaint"] / 30.0),
    0.0,
)

complaint_agg["complaint_recent_event_risk_raw"] = complaint_agg[
    "complaint_recent_event_risk_raw"
].clip(0, 1)

complaint_agg.head(3)

,user_id,complaint_reporter_count,complaint_reporter_open_count,complaint_reporter_escalated_count,complaint_reporter_resolved_count,complaint_reporter_severity_sum,complaint_reporter_last_at,complaint_target_count,complaint_target_open_count,complaint_target_escalated_count,...,complaint_target_last_at,complaint_total_count,complaint_open_count,complaint_escalated_count,complaint_resolved_count,complaint_total_severity,last_complaint_at,days_since_last_complaint_raw,days_since_last_complaint,complaint_recent_event_risk_raw
0,1,1.0,0.0,0.0,1.0,0.35,2025-03-03 15:04:17+00:00,2.0,1.0,0.0,...,2023-05-02 19:10:27+00:00,3.0,1.0,0.0,2.0,2.260,2025-03-03 15:04:17+00:00,403.372025,403.372025,0.069225
1,9,0.0,0.0,0.0,0.0,0.00,NaT,2.0,2.0,0.0,...,2023-09-30 21:30:24+00:00,2.0,2.0,0.0,0.0,2.760,2023-09-30 21:30:24+00:00,923.103889,923.103889,0.031476
2,10,3.0,2.0,0.0,0.0,2.01,2023-07-17 03:03:13+00:00,2.0,1.0,0.0,...,2024-11-25 01:55:08+00:00,5.0,3.0,0.0,1.0,3.504,2024-11-25 01:55:08+00:00,501.920046,501.920046,0.056399


### Validation des features réclamations

Les agrégats réclamations sont cohérents et la séparation reporter / target fonctionne.

La récence a été rendue plus progressive pour éviter une saturation trop rapide du signal. Cela permet de conserver une vraie nuance entre une réclamation récente et une réclamation ancienne dans le score final.

## Fusion des features

On rassemble maintenant tous les signaux au niveau utilisateur.

Les colonnes temporelles sont reconverties explicitement en datetime UTC avant les calculs pour éviter les erreurs de type.

In [7]:
features = users[[
    "id",
    "email",
    "country",
    "signup_at_utc",
    "last_seen_at_utc",
    "status",
    "status_is_anomalous",
]].copy().rename(columns={"id": "user_id"})

features["signup_at_utc"] = pd.to_datetime(features["signup_at_utc"], utc=True, errors="coerce")
features["last_seen_at_utc"] = pd.to_datetime(features["last_seen_at_utc"], utc=True, errors="coerce")

for df in [payment_agg, membership_agg, complaint_agg]:
    features = features.merge(df, on="user_id", how="left")

for col in [
    "payment_last_at",
    "last_membership_end_at",
    "last_complaint_at",
    "complaint_reporter_last_at",
    "complaint_target_last_at",
]:
    if col in features.columns:
        features[col] = pd.to_datetime(features[col], utc=True, errors="coerce")

numeric_cols = features.select_dtypes(include=["number"]).columns.tolist()
for col in numeric_cols:
    features[col] = features[col].fillna(0)

features["account_age_days"] = days_between(
    pd.Series(reference_date, index=features.index),
    features["signup_at_utc"],
)
features["account_age_days"] = clip_days(features["account_age_days"], 3650)

last_seen_ref = features["last_seen_at_utc"].fillna(features["signup_at_utc"]).fillna(reference_date)
features["days_since_last_seen"] = days_between(
    pd.Series(reference_date, index=features.index),
    last_seen_ref,
)
features["days_since_last_seen"] = clip_days(features["days_since_last_seen"], 365)

features["is_new_user"] = (features["account_age_days"] < 30).astype(int)
features["inactive_6m_flag"] = (features["days_since_last_seen"] >= 180).astype(int)
features["has_history_flag"] = (
    (features["payment_total_count"] + features["membership_total_count"] + features["complaint_total_count"]) > 0
).astype(int)

features.head(3)

,user_id,email,country,signup_at_utc,last_seen_at_utc,status,status_is_anomalous,payment_total_count,payment_success_count,payment_failed_count,...,complaint_total_severity,last_complaint_at,days_since_last_complaint_raw,days_since_last_complaint,complaint_recent_event_risk_raw,account_age_days,days_since_last_seen,is_new_user,inactive_6m_flag,has_history_flag
0,1,user_1@yahoo.fr,IT,2021-02-09 21:26:47+00:00,2021-04-28 14:23:00+00:00,1.0,0,2.0,0.0,2.0,...,2.26,2025-03-03 15:04:17+00:00,403.372025,403.372025,0.069225,1886.106400,365.0,0,1,1
1,2,user_2@gmail.com,CH,2022-01-09 06:49:44+00:00,2023-01-05 17:03:20+00:00,0.0,0,0.0,0.0,0.0,...,0.00,NaT,0.000000,0.000000,0.000000,1552.715463,365.0,0,1,0
2,3,user_3@outlook.com,BE,2020-03-21 04:34:57+00:00,2021-07-30 05:38:37+00:00,1.0,0,0.0,0.0,0.0,...,0.00,NaT,0.000000,0.000000,0.000000,2211.809062,365.0,0,1,0


### Validation de la fusion des features

La fusion des signaux au niveau utilisateur est cohérente.

Les colonnes temporelles sont bien converties, les agrégats comportementaux sont présents, et les flags de récence fonctionnent. Le plafonnement à 365 jours sur `days_since_last_seen` reste volontaire pour éviter qu’une ancienneté extrême écrase le score.

## Calcul du score

Le score final est une heuristique opérationnelle, pas un modèle supervisé.

On combine des composantes de paiement, de réclamation, de churn et de récence, puis on ramène le résultat sur 100.

In [8]:
# Composantes brutes du risque
features["payment_inactivity_risk_raw"] = clip_days(features["days_since_last_payment"], 365) / 365.0
features["membership_recent_exit_risk_raw"] = np.where(
    features["membership_exit_count"] > 0,
    1 - (features["days_since_last_membership_end"] / 365.0),
    0.0
)
features["membership_recent_exit_risk_raw"] = pd.Series(
    features["membership_recent_exit_risk_raw"],
    index=features.index
).clip(0, 1)

features["payment_risk_raw"] = (
    np.log1p(features["payment_failed_count"])
    + 1.5 * np.log1p(features["payment_disputed_count"])
    + 0.8 * np.log1p(features["payment_refunded_count"])
    + 0.3 * np.log1p(features["payment_pending_count"])
    + 0.2 * np.log1p(features["payment_canceled_count"])
    + 2.0 * features["payment_issue_rate"]
    + 0.5 * features["payment_inactivity_risk_raw"]
)

features["complaint_risk_raw"] = (
    1.2 * np.log1p(features["complaint_open_count"])
    + 1.5 * np.log1p(features["complaint_escalated_count"])
    + 0.8 * np.log1p(features["complaint_reporter_severity_sum"])
    + 1.2 * np.log1p(features["complaint_target_severity_sum"])
    + 0.8 * features["complaint_recent_event_risk_raw"]
)

features["membership_risk_raw"] = (
    1.0 * np.log1p(features["membership_exit_count"])
    + 0.7 * np.log1p(features["short_membership_count"])
    + 0.3 * np.log1p(features["distinct_subscription_count"])
    + 0.2 * np.log1p(features["distinct_brand_count"])
    + 0.5 * features["membership_churn_rate"]
    + 0.3 * features["brand_switch_rate"]
    + 0.7 * features["membership_recent_exit_risk_raw"]
)

features["recency_risk_raw"] = features["days_since_last_seen"] / 365.0

# Normalisation robuste
features["payment_risk_norm"] = robust_minmax(features["payment_risk_raw"])
features["complaint_risk_norm"] = robust_minmax(features["complaint_risk_raw"])
features["membership_risk_norm"] = robust_minmax(features["membership_risk_raw"])
features["recency_risk_norm"] = robust_minmax(features["recency_risk_raw"])

# Score final sur 100
features["risk_score"] = (
    100 * (
        0.40 * features["payment_risk_norm"]
        + 0.30 * features["complaint_risk_norm"]
        + 0.20 * features["membership_risk_norm"]
        + 0.10 * features["recency_risk_norm"]
    )
).round(2)

# Tiers lisibles
features["risk_tier"] = pd.cut(
    features["risk_score"],
    bins=[-0.01, 24.99, 49.99, 74.99, 100.0],
    labels=["low", "watch", "high", "critical"]
).astype(str)

features["risk_tier"] = features["risk_tier"].replace("nan", "low")

features[[
    "user_id",
    "risk_score",
    "risk_tier",
    "payment_risk_norm",
    "complaint_risk_norm",
    "membership_risk_norm",
    "recency_risk_norm",
]].head(10)

,user_id,risk_score,risk_tier,payment_risk_norm,complaint_risk_norm,membership_risk_norm,recency_risk_norm
0,1,67.07,high,0.795697,0.642385,0.298536,1.000000
1,2,10.00,low,0.000000,0.000000,0.000000,1.000000
2,3,10.00,low,0.000000,0.000000,0.000000,1.000000
3,4,10.00,low,0.000000,0.000000,0.000000,1.000000
4,5,0.00,low,0.000000,0.000000,0.000000,0.000000
5,6,14.42,low,0.110556,0.000000,0.000000,1.000000
6,7,8.92,low,0.000000,0.000000,0.000000,0.891523
7,8,10.00,low,0.000000,0.000000,0.000000,1.000000
8,9,25.11,watch,0.000000,0.776503,0.000000,0.181704
9,10,75.69,critical,0.950933,1.000000,0.147953,0.469369


### Validation du score

Le score final est cohérent et exploitable.

Il est bien borné entre 0 et 100, les tiers de risque sont lisibles, et les utilisateurs avec davantage de signaux incidents remontent correctement dans le classement. Le score reflète donc les choix méthodologiques retenus : paiements, réclamations, memberships et récence contribuent au résultat final.

## Export du CSV scoré

Le résultat final du scoring est exporté en CSV pour répondre à la contrainte du test.

Le fichier conserve les colonnes utiles à l’opérationnel, le score final, le rang de risque et les signaux principaux qui permettent d’expliquer la priorisation.

In [ ]:
# Colonnes finales à conserver
final_cols = [
    "user_id",
    "email",
    "country",
    "signup_at_utc",
    "last_seen_at_utc",
    "status",
    "status_is_anomalous",
    "is_new_user",
    "inactive_6m_flag",
    "has_history_flag",
    "payment_total_count",
    "payment_success_count",
    "payment_failed_count",
    "payment_disputed_count",
    "payment_refunded_count",
    "payment_pending_count",
    "payment_canceled_count",
    "payment_issue_rate",
    "payment_distinct_subscription_count",
    "payment_distinct_brand_count",
    "membership_total_count",
    "membership_exit_count",
    "membership_current_count",
    "short_membership_count",
    "avg_membership_duration_days",
    "median_membership_duration_days",
    "distinct_subscription_count",
    "distinct_brand_count",
    "membership_churn_rate",
    "brand_switch_rate",
    "complaint_total_count",
    "complaint_reporter_count",
    "complaint_target_count",
    "complaint_open_count",
    "complaint_escalated_count",
    "complaint_resolved_count",
    "complaint_total_severity",
    "days_since_last_payment_raw",
    "days_since_last_payment",
    "days_since_last_complaint_raw",
    "days_since_last_complaint",
    "days_since_last_membership_end_raw",
    "days_since_last_membership_end",
    "days_since_last_seen",
    "payment_risk_raw",
    "complaint_risk_raw",
    "membership_risk_raw",
    "recency_risk_raw",
    "payment_risk_norm",
    "complaint_risk_norm",
    "membership_risk_norm",
    "recency_risk_norm",
    "risk_score",
    "risk_tier",
]

# Vérification explicite des colonnes avant export
missing_cols = [c for c in final_cols if c not in features.columns]
assert not missing_cols, f"Colonnes manquantes dans features : {missing_cols}"

# Construction du CSV final
final_df = features[final_cols].copy()
final_df = final_df.sort_values("risk_score", ascending=False).reset_index(drop=True)
final_df["risk_rank"] = np.arange(1, len(final_df) + 1)

# Ordre final des colonnes, avec risk_rank à la fin
final_cols_output = final_cols + ["risk_rank"]
final_df = final_df[final_cols_output]

# Export final
output_path = Path("outputs/subscribers_risk_scored.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
final_df.to_csv(output_path, index=False)

output_path

WindowsPath('H:/Ecole/ECE/CESURE/ALTERNANCE/Sharesub.com/TEST_TECHNIQUE/risk-monitor/outputs/subscribers_risk_scored.csv')

### Validation de l'export CSV

Le CSV scoré a été généré avec succès.

Il contient les colonnes utiles au suivi opérationnel, le score final, le rang de risque et les indicateurs nécessaires pour interpréter la priorisation des subscribers.

## Validation du scoring

On vérifie maintenant que :
- le score est bien borné entre 0 et 100 ;
- les tiers de risque sont cohérents ;
- les nouveaux users sans historique ne sont pas artificiellement sur-scorés ;
- les cas d’inactivité longue remontent bien dans le score ;
- le CSV final a bien été généré.

In [10]:
# Contrôle de base
assert final_df["risk_score"].between(0, 100).all(), "Le score doit être compris entre 0 et 100"

print(final_df["risk_tier"].value_counts(dropna=False))
print(final_df["risk_score"].describe())

# Cas utiles pour contrôle visuel
display(final_df.head(10))
display(final_df[final_df["is_new_user"] == 1].head(10))
display(final_df[final_df["inactive_6m_flag"] == 1].head(10))

risk_tier
low         1072
watch        405
high         374
critical     150
Name: count, dtype: int64
count    2001.000000
mean       31.363308
std        25.140023
min         0.000000
25%        10.000000
50%        23.680000
75%        50.840000
max       100.000000
Name: risk_score, dtype: float64


,user_id,email,country,signup_at_utc,last_seen_at_utc,status,status_is_anomalous,is_new_user,inactive_6m_flag,has_history_flag,...,complaint_risk_raw,membership_risk_raw,recency_risk_raw,payment_risk_norm,complaint_risk_norm,membership_risk_norm,recency_risk_norm,risk_score,risk_tier,risk_rank
0,538,user_538@gmail.com,FR,2026-12-26 16:13:19+00:00,2021-08-21 16:26:48+00:00,4.0,0,1,1,1,...,4.849201,2.827656,1.0,1.000000,1.000000,1.000000,1.0,100.00,critical,1
1,678,user_678@hotmail.com,AT,2024-08-30 23:37:15+00:00,2024-11-30 16:23:03+00:00,1.0,0,0,1,1,...,4.080611,2.342453,1.0,1.000000,1.000000,1.000000,1.0,100.00,critical,2
2,621,user_621@outlook.com,PT,2021-07-12 04:03:59+00:00,2023-09-30 23:24:19+00:00,0.0,0,0,1,1,...,3.945480,2.342453,1.0,1.000000,1.000000,1.000000,1.0,100.00,critical,3
3,1399,user_1399@proton.me,IT,2023-12-13 19:15:23+00:00,2025-02-19 11:40:09+00:00,0.0,0,0,1,1,...,5.048661,2.452961,1.0,1.000000,1.000000,1.000000,1.0,100.00,critical,4
4,1702,user_1702@free.fr,BE,2020-09-26 01:42:50+00:00,2023-11-14 07:12:57+00:00,0.0,0,0,1,1,...,3.519886,2.326301,1.0,1.000000,0.984504,0.993105,1.0,99.40,critical,5
5,1316,user_1316@gmail.com,FR,2022-04-03 15:21:01+00:00,2024-03-10 08:53:52+00:00,0.0,0,0,1,1,...,3.397228,2.342453,1.0,1.000000,0.950196,1.000000,1.0,98.51,critical,6
6,1132,user_1132@hotmail.com,PT,2020-09-23 01:36:16+00:00,2021-09-07 07:32:42+00:00,-1.0,1,0,1,1,...,3.606401,2.342453,1.0,0.944356,1.000000,1.000000,1.0,97.77,critical,7
7,472,user_472@gmail.com,PT,2021-05-24 06:42:26+00:00,2022-11-30 23:58:47+00:00,2.0,0,0,1,1,...,3.282531,2.342453,1.0,1.000000,0.918116,1.000000,1.0,97.54,critical,8
8,1272,user_1272@outlook.com,DE,2021-10-15 10:28:57+00:00,2021-12-27 11:11:18+00:00,0.0,0,0,1,1,...,4.114743,2.827656,1.0,0.921871,1.000000,1.000000,1.0,96.87,critical,9
9,60,user_60@hotmail.com,BE,2021-09-27 05:46:23+00:00,2023-05-15 08:06:11+00:00,0.0,0,0,1,1,...,5.722205,2.639147,1.0,0.913702,1.000000,1.000000,1.0,96.55,critical,10


,user_id,email,country,signup_at_utc,last_seen_at_utc,status,status_is_anomalous,is_new_user,inactive_6m_flag,has_history_flag,...,complaint_risk_raw,membership_risk_raw,recency_risk_raw,payment_risk_norm,complaint_risk_norm,membership_risk_norm,recency_risk_norm,risk_score,risk_tier,risk_rank
0,538,user_538@gmail.com,FR,2026-12-26 16:13:19+00:00,2021-08-21 16:26:48+00:00,4.0,0,1,1,1,...,4.849201,2.827656,1.000000,1.000000,1.000000,1.000000,1.000000,100.00,critical,1
181,398,user_398@proton.me,CH,2026-11-20 04:27:08+00:00,2024-08-08 14:56:28+00:00,4.0,0,1,1,1,...,1.952798,1.539721,1.000000,0.796242,0.546193,0.657311,1.000000,71.38,high,182
293,11,user_11@yahoo.fr,AT,2026-10-08 10:23:00+00:00,2023-07-14 04:17:17+00:00,0.0,0,1,1,1,...,1.937556,0.346574,1.000000,0.864787,0.541930,0.147953,1.000000,63.81,high,294
577,339,user_339@free.fr,IT,2026-05-18 22:08:06+00:00,2025-03-06 16:50:15+00:00,0.0,0,1,1,1,...,1.727381,0.346574,1.000000,0.471721,0.483144,0.147953,1.000000,46.32,watch,578
604,1145,user_1145@free.fr,BE,2026-04-08 14:33:29+00:00,2025-03-19 02:03:42+00:00,0.0,0,1,1,1,...,0.397336,0.346574,1.000000,0.714827,0.111134,0.147953,1.000000,44.89,watch,605
720,850,user_850@gmail.com,DE,2026-12-29 09:04:30+00:00,2025-11-13 23:32:28+00:00,4.0,0,1,0,1,...,1.718554,0.346574,0.405532,0.404797,0.480675,0.147953,0.306962,36.64,watch,721
966,1645,user_1645@proton.me,,2026-12-15 02:54:39+00:00,2024-11-16 08:30:09+00:00,2.0,0,1,1,1,...,1.731951,0.000000,1.000000,0.000000,0.484423,0.000000,1.000000,24.53,low,967
1050,229,user_229@hotmail.com,PT,2026-03-25 10:02:41+00:00,2025-08-06 14:19:44+00:00,0.0,0,1,1,1,...,1.843606,0.000000,0.677816,0.000000,0.515652,0.000000,0.624394,21.71,low,1051
1397,1861,user_1861@hotmail.com,DE,2026-08-14 02:41:00+00:00,2021-12-17 11:37:23+00:00,0.0,0,1,1,0,...,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,10.00,low,1398
1508,698,user_698@yahoo.fr,AT,2026-10-16 14:36:29+00:00,2021-04-18 19:45:24+00:00,0.0,0,1,1,0,...,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,10.00,low,1509


,user_id,email,country,signup_at_utc,last_seen_at_utc,status,status_is_anomalous,is_new_user,inactive_6m_flag,has_history_flag,...,complaint_risk_raw,membership_risk_raw,recency_risk_raw,payment_risk_norm,complaint_risk_norm,membership_risk_norm,recency_risk_norm,risk_score,risk_tier,risk_rank
0,538,user_538@gmail.com,FR,2026-12-26 16:13:19+00:00,2021-08-21 16:26:48+00:00,4.0,0,1,1,1,...,4.849201,2.827656,1.0,1.000000,1.000000,1.000000,1.0,100.00,critical,1
1,678,user_678@hotmail.com,AT,2024-08-30 23:37:15+00:00,2024-11-30 16:23:03+00:00,1.0,0,0,1,1,...,4.080611,2.342453,1.0,1.000000,1.000000,1.000000,1.0,100.00,critical,2
2,621,user_621@outlook.com,PT,2021-07-12 04:03:59+00:00,2023-09-30 23:24:19+00:00,0.0,0,0,1,1,...,3.945480,2.342453,1.0,1.000000,1.000000,1.000000,1.0,100.00,critical,3
3,1399,user_1399@proton.me,IT,2023-12-13 19:15:23+00:00,2025-02-19 11:40:09+00:00,0.0,0,0,1,1,...,5.048661,2.452961,1.0,1.000000,1.000000,1.000000,1.0,100.00,critical,4
4,1702,user_1702@free.fr,BE,2020-09-26 01:42:50+00:00,2023-11-14 07:12:57+00:00,0.0,0,0,1,1,...,3.519886,2.326301,1.0,1.000000,0.984504,0.993105,1.0,99.40,critical,5
5,1316,user_1316@gmail.com,FR,2022-04-03 15:21:01+00:00,2024-03-10 08:53:52+00:00,0.0,0,0,1,1,...,3.397228,2.342453,1.0,1.000000,0.950196,1.000000,1.0,98.51,critical,6
6,1132,user_1132@hotmail.com,PT,2020-09-23 01:36:16+00:00,2021-09-07 07:32:42+00:00,-1.0,1,0,1,1,...,3.606401,2.342453,1.0,0.944356,1.000000,1.000000,1.0,97.77,critical,7
7,472,user_472@gmail.com,PT,2021-05-24 06:42:26+00:00,2022-11-30 23:58:47+00:00,2.0,0,0,1,1,...,3.282531,2.342453,1.0,1.000000,0.918116,1.000000,1.0,97.54,critical,8
8,1272,user_1272@outlook.com,DE,2021-10-15 10:28:57+00:00,2021-12-27 11:11:18+00:00,0.0,0,0,1,1,...,4.114743,2.827656,1.0,0.921871,1.000000,1.000000,1.0,96.87,critical,9
9,60,user_60@hotmail.com,BE,2021-09-27 05:46:23+00:00,2023-05-15 08:06:11+00:00,0.0,0,0,1,1,...,5.722205,2.639147,1.0,0.913702,1.000000,1.000000,1.0,96.55,critical,10


### Validation finale du scoring

Le score final est cohérent et exploitable.

Il est bien borné entre 0 et 100, les tiers de risque sont lisibles, et les utilisateurs avec davantage de signaux incidents remontent correctement dans le classement. La distribution des scores est suffisamment étalée pour permettre une priorisation opérationnelle.

Une réserve reste à documenter : certaines dates d’inscription apparaissent dans le futur par rapport aux autres signaux de la base. Le score reste utilisable, mais ce point doit être mentionné comme limite de qualité des données.

## Synthèse de l’étape 3

Le score de risque a été construit à partir de la base nettoyée de l’étape 2, avec une logique déterministe et reproductible.

Décisions principales :
- les signaux utilisés sont les paiements, les réclamations, les memberships et la récence d’activité ;
- les colonnes brutes de l’étape 2 ont servi de base, sans réintroduire les anomalies de nettoyage ;
- la date de référence a été recalculée à partir des signaux opérationnels uniquement, afin d’éviter qu’une date d’inscription future anormale ne déforme le scoring ;
- les agrégats orphelins ont été exclus des features finales ;
- les composantes de risque ont été normalisées avant d’être combinées en un score sur 100 ;
- le score final a été traduit en tiers lisibles : `low`, `watch`, `high`, `critical` ;
- le résultat a été exporté en CSV.

Le fichier final est :
`outputs/subscribers_risk_scored.csv`

Limite à documenter :
- certaines dates d’inscription apparaissent dans le futur par rapport aux autres signaux de la base ; le score reste exploitable, mais ce point doit être mentionné dans les limites connues du projet.